# Speech Review — Colab 工作台

保留现有 Whisper 原 notebook 流程、原 TXT 参考、人工修订、批量保存、MFA 强制对齐、波形边界编辑及真实 Praat TextGrid 导出。

**此版本使用 Colab notebook 内嵌工作台，不发布公共网站。请仅在允许 Web UI 的运行环境中使用：Colab Pro 等付费计划且有可用 compute units，或你控制的合适运行时。** 当前免费托管运行时限制主要经 Web UI 操作；不要使用隧道、保活或其他方式绕过限制。

程序代码已经嵌入此 notebook，不包含你的录音、修订、账号或本地研究数据。你需要自行上传录音 ZIP，或选择授权访问自己的 Google Drive 文件夹。代码执行与录音处理发生在 Google 运行时，不在 iPad/电脑本机。分享 notebook 不等于分享同一个正在运行的工作台；每个人通常运行自己的会话。

从第 1 步依次运行。可选择 GPU（Runtime → Change runtime type），CPU 也能运行，但更慢。首次下载模型和 MFA 需要时间。请先用一个受试者试运行。

目前已完成本地测试；此 notebook 尚未在你的 Colab 账号或 iPad 上实际运行验收。


## 1. 解包程序代码（无需另传项目 ZIP）

In [ ]:
import base64,hashlib,io,zipfile,sys
from pathlib import Path

_bundle = base64.b64decode("...")
assert hashlib.sha256(_bundle).hexdigest() == "3e5b8d1b7beb1e271403a54796a0771990486475aecfb6eb865546a3aa6ef09c", "程序包校验失败"
APP_ROOT = Path("/content/speech_review_code/3e5b8d1b7beb")
APP_ROOT.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(io.BytesIO(_bundle)) as archive:
    for item in archive.infolist():
        destination = (APP_ROOT/item.filename).resolve()
        if APP_ROOT.resolve() not in destination.parents:
            raise ValueError("程序包路径越界")
    archive.extractall(APP_ROOT)
if str(APP_ROOT) not in sys.path:sys.path.insert(0,str(APP_ROOT))
print("程序已准备：",APP_ROOT)


## 2. 安装 Python 依赖
安装输出直接显示在本单元；不需要 VS Code 或本机 Python。

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,"-m","pip","install","-r",str(APP_ROOT/"requirements.txt")],check=True)
print("Python 依赖已安装")


## 3. 保存位置：临时会话或 Google Drive

`PERSIST_TO_DRIVE=True` 会请求你手动授权 Drive，并把输入、修订与人工边界保存在 `MyDrive/SpeechReviewColab`。模型、解码缓存及 MFA 临时计算仍放在 Colab 本地磁盘。

默认 `False` 不访问 Drive；**Colab 运行时被回收后，临时文件会丢失，结束前请导出。** 运行时及 GPU 资源不保证持续可用。


In [ ]:
#@title 保存位置
PERSIST_TO_DRIVE = False #@param {type:"boolean"}
import os
SESSION_ROOT=Path("/content/speech_review_session")
if PERSIST_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SAVE_ROOT=Path("/content/drive/MyDrive/SpeechReviewColab")
else:
    SAVE_ROOT=SESSION_ROOT
SAVE_ROOT.mkdir(parents=True,exist_ok=True)
SESSION_ROOT.mkdir(parents=True,exist_ok=True)
os.environ["SPEECH_REVIEW_DATA"]=str(SAVE_ROOT/"projects")
os.environ["SPEECH_REVIEW_ALIGNMENT_WORK"]=str(SESSION_ROOT/"alignment_work")
os.environ["SPEECH_REVIEW_PLAYBACK_CACHE"]=str(SESSION_ROOT/"playback_cache")
os.environ["MFA_ROOT_DIR"]=str(SESSION_ROOT/"mfa_models")
os.environ["HF_HOME"]=str(SESSION_ROOT/"whisper_models")
os.environ["PYTHONUTF8"]="1"
print("修订和边界保存目录：",SAVE_ROOT/"projects")
print("此目录持久化：",PERSIST_TO_DRIVE)


## 4. 安装独立 Linux MFA 环境
采用与桌面版相同的 MFA 3.3.10，以及 mandarin_mfa / mandarin_china_mfa。约数百 MB，首次可能较慢。

In [ ]:
subprocess.run([sys.executable,str(APP_ROOT/"install_mfa.py")],cwd=APP_ROOT,check=True)
print("MFA 运行环境已安装；普通话模型在首次对齐时自动下载。")


## 5. 检查 GPU；如使用 GPU，准备兼容的 CUDA 库
模型选择沿用原 notebook：Number/Homo 优先 medium GPU；Non-homo 优先 large-v3 GPU；不可用时回退 medium CPU。

In [ ]:
import shutil,site
GPU_AVAILABLE=False
if shutil.which("nvidia-smi"):
    gpu_check=subprocess.run(["nvidia-smi","--query-gpu=name","--format=csv,noheader"],capture_output=True,text=True)
    GPU_AVAILABLE=gpu_check.returncode==0 and bool(gpu_check.stdout.strip())
    if GPU_AVAILABLE:print("GPU：",gpu_check.stdout.strip())
if GPU_AVAILABLE:
    subprocess.run([sys.executable,"-m","pip","install","nvidia-cublas-cu12","nvidia-cudnn-cu12"],check=True)
    cuda_dirs=[]
    for directory in site.getsitepackages():
        cuda_dirs.extend(str(p) for p in Path(directory).glob("nvidia/*/lib") if p.is_dir())
    os.environ["LD_LIBRARY_PATH"]=os.pathsep.join(cuda_dirs+[os.environ.get("LD_LIBRARY_PATH","")])
else:
    print("当前无 GPU，将使用 CPU。")


## 6. 启动后台服务（仅当前 Colab 账号代理）

程序绑定运行时内部的 127.0.0.1，通过 Colab 自带代理内嵌访问。这里不创建公开 tunnel，不输出可供他人共用的公开网址。


In [ ]:
import json,time,urllib.request,urllib.error
from urllib.parse import urlsplit
from google.colab import output
PORT=8765
proxy_url=output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
proxy=urlsplit(proxy_url)
if proxy.scheme!="https" or not proxy.hostname:
    raise RuntimeError("未获得有效的 Colab 代理地址，请确认在 Colab 页面内运行。")
os.environ["SPEECH_REVIEW_PROXY_ORIGIN"]=f"https://{proxy.netloc}"
if "REVIEW_PROCESS" not in globals() or REVIEW_PROCESS.poll() is not None:
    REVIEW_LOG_PATH=SESSION_ROOT/"server.log"
    REVIEW_LOG=REVIEW_LOG_PATH.open("a",encoding="utf-8")
    REVIEW_PROCESS=subprocess.Popen([sys.executable,"-m","uvicorn","backend.main:app","--host","127.0.0.1","--port",str(PORT)],cwd=APP_ROOT,env=os.environ.copy(),stdout=REVIEW_LOG,stderr=subprocess.STDOUT)
BASE=f"http://127.0.0.1:{PORT}/api"
def api(path,payload=None):
    data=None if payload is None else json.dumps(payload,ensure_ascii=False).encode()
    request=urllib.request.Request(BASE+path,data=data,headers={"Content-Type":"application/json"})
    try:
        with urllib.request.urlopen(request,timeout=90) as response:return json.load(response)
    except urllib.error.HTTPError as exc:
        raise RuntimeError(exc.read().decode("utf-8",errors="replace")) from exc
for attempt in range(60):
    try:
        print("后台就绪：",api("/alignment/status"));break
    except Exception:
        if REVIEW_PROCESS.poll() is not None or attempt==59:
            print(REVIEW_LOG_PATH.read_text(encoding="utf-8",errors="replace")[-6000:])
            raise RuntimeError("服务启动失败，请检查上面的日志。")
        time.sleep(1)


## 7. 载入录音：上传 ZIP 或填写 Drive 文件夹

ZIP 内放 WAV 与同名 TXT，可包含受试者子文件夹。不是上传本机文件夹路径（例如 `C:\Users\...`）；Colab 无法读取你电脑的磁盘。

如已挂载 Drive，可填写直接包含 WAV/TXT 的 `/content/drive/MyDrive/...` 路径，避免重新上传。默认不改变原始 WAV/TXT。自动识别 Number/Homo/Non-homo，也可手动指定。


In [ ]:
#@title 导入受试者
AUDIO_FOLDER = "" #@param {type:"string"}
CONDITION = "unknown" #@param ["unknown","number","homo","nonhomo"]
from colab_support import extract_inputs
if AUDIO_FOLDER.strip():
    candidates=[Path(AUDIO_FOLDER.strip()).expanduser().resolve()]
    if not candidates[0].is_dir():raise ValueError("音频文件夹不存在，请检查 Drive 挂载和路径。")
else:
    from google.colab import files
    uploaded=files.upload()
    names=[name for name in uploaded if name.lower().endswith(".zip")]
    if len(names)!=1:raise ValueError("每次请选择一个录音 ZIP 文件。")
    name=names[0]
    raw=uploaded[name]
    archive_key=hashlib.sha256(raw).hexdigest()[:16]
    archive_path=SESSION_ROOT/(archive_key+".zip")
    archive_path.write_bytes(raw)
    candidates=extract_inputs(archive_path,SAVE_ROOT/"inputs"/archive_key)
    del uploaded,raw
if not candidates:raise ValueError("未找到 WAV 文件。")
if len(candidates)==1:
    imported=api("/import",{"folder":str(candidates[0]),"condition":CONDITION})
    print("已导入：",imported["participant"],"，录音数：",len(imported["trials"]))
else:
    print("检测到多个受试者，请在工作台中逐个复制以下目录并导入：")
    for folder in candidates:print(folder)


## 8. 打开完整工作台

在下方内嵌界面操作：Whisper → 复听／修改 → 一键保存 TXT → MFA 对齐 → “对齐与边界编辑” → 保存人工边界／导出。

这里使用同一页面内的标签切换。由于 Colab 浏览器存储隔离，不提供独立新窗口链接。若 iframe 被浏览器阻止，查看浏览器对 Colab/googleusercontent 的相关提示；不关闭安全功能或使用公共隧道绕过。

**iPad 浏览器的触控拖动尚待实机验证；必要时可用开始／结束时间输入框精确改边界。**


In [ ]:
output.serve_kernel_port_as_iframe(PORT,path="/?embedded=1&view=transcription",height=1100,cache_in_notebook=False)


## 9. 可选：从 notebook 下载全部项目结果

如果 iPad 的 iframe 下载不方便，完成保存和对齐后，把下面开关改为 True 再运行。只导出已保存文本和有效对齐，自动／人工版本分开。不会把输入录音、模型、账号信息放进导出 ZIP。


In [ ]:
#@title 下载全部结果
DOWNLOAD_NOW = False #@param {type:"boolean"}
if DOWNLOAD_NOW:
    from google.colab import files
    projects=api("/projects")
    if not projects:raise ValueError("还没有导入项目。")
    result_path=SESSION_ROOT/"speech_review_results.zip"
    with zipfile.ZipFile(result_path,"w",zipfile.ZIP_DEFLATED) as archive:
        for project in projects:
            with urllib.request.urlopen(BASE+f"/projects/{project['id']}/export-everything",timeout=120) as response:
                data=response.read()
            archive.writestr(f"{project['id']}/participant_export.zip",data)
    files.download(str(result_path))
else:
    print("尚未下载；需要时设 DOWNLOAD_NOW=True 并重新运行此单元。")


## 10. 可选：查看日志／结束运行
不使用自动保活；关闭运行时前保存并导出。

In [ ]:
#@title 日志与停止
SHOW_LOG = False #@param {type:"boolean"}
STOP_NOW = False #@param {type:"boolean"}
if SHOW_LOG:print(REVIEW_LOG_PATH.read_text(encoding="utf-8",errors="replace")[-7000:])
if STOP_NOW and "REVIEW_PROCESS" in globals() and REVIEW_PROCESS.poll() is None:
    REVIEW_PROCESS.terminate()
    REVIEW_PROCESS.wait(timeout=30)
    print("已停止本 notebook 启动的服务。")


### 参考与限制

- [Colab 官方 FAQ：运行限制、Pro 与 Education 资格、数据和会话生命周期](https://research.google.com/colaboratory/faq.html)
- [Google 官方端口嵌入实现：iframe；旧的独立窗口方法已弃用](https://github.com/googlecolab/colabtools/blob/main/google/colab/output/_util.py)
- [MFA 安装说明](https://montreal-forced-aligner.readthedocs.io/en/v3.3.2/installation.html)

持久化项目的数据结构与本地版相同，但不能直接使用原电脑的音频路径。这里会重新导入上传／Drive 的音频；迁移旧人工边界可导入对应 reviewed.TextGrid。分享 notebook 前请清除包含研究材料的运行输出。是否有 Pro 或可用额度，以账号 Colab 设置和 Resources 面板为准；本 notebook 不访问你的订阅或付款设置。
